# Occlusion Sensitivity Analysis
**Genomic-RawSeq-Analyzer — Semester 2**

Aggregates occlusion importance scores across **100+ high-confidence cancer reads**,
identifies the top recurring k-mer motifs, cross-references with COSMIC mutation
signatures, and produces a population-level heatmap.

**Outputs saved to Google Drive:**
- `results/occlusion/occlusion_heatmap.png`
- `results/occlusion/aggregated_importance.png`
- `results/occlusion/motifs.json`  ← used by LLMReports.ipynb

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/492'
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q tensorflow seaborn matplotlib scikit-learn

In [ ]:
import sys
sys.path.insert(0, 'src')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json

from tensorflow.keras.models import load_model
from data_loader import DataLoader
from explainability import OcclusionAnalyzer

BATCH_DIR  = 'results/batches'
MODEL_PATH = 'results/cnn_baseline.keras'
SAVE_DIR   = 'results/occlusion'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Imports OK')

## Step 1 — Load Model & Cancer Reads

In [ ]:
print('Loading model...')
model = load_model(MODEL_PATH)

print('Loading batch data...')
X, y, run_ids = DataLoader.load_all_batches(BATCH_DIR)
X_cancer = X[y == 1]
print(f'Total cancer reads: {len(X_cancer):,}')

## Step 2 — Individual Saliency Maps (Top-3 Most Confident Reads)

In [ ]:
analyzer = OcclusionAnalyzer(model)

saliency_dir = f'{SAVE_DIR}/saliency_maps'
os.makedirs(saliency_dir, exist_ok=True)

print('Finding top-3 most confident cancer reads from 2000 samples...')
analyzer.plot_top_cancer_reads(
    X_cancer,
    top_n=3,
    sample_size=2000,
    save_dir=saliency_dir,
)
print('Individual saliency maps saved.')

## Step 3 — Population-Level Aggregation (500 reads, k=5)

In [ ]:
# ── Configuration ──────────────────────────────────────────
N_SAMPLES  = 500    # number of cancer reads to analyse
K          = 5      # k-mer length
TOP_KMERS  = 5      # motifs to report
THRESHOLD  = 0.60   # minimum cancer confidence

print(f'Running full analysis: {N_SAMPLES} reads, k={K}...')
results = analyzer.run_full_analysis(
    X, y,
    n_samples=N_SAMPLES,
    k=K,
    top_kmers=TOP_KMERS,
    confidence_threshold=THRESHOLD,
    save_dir=SAVE_DIR,
)

print(f'\nReads used   : {results["n_reads_used"]}')
print(f'Heatmap      : {SAVE_DIR}/occlusion_heatmap.png')
print(f'Motifs JSON  : {SAVE_DIR}/motifs.json')

## Step 4 — Top Motifs with COSMIC Cross-Reference

In [ ]:
print('\n' + '='*65)
print('TOP K-MER MOTIFS AND COSMIC SIGNATURE ASSOCIATIONS')
print('='*65)

for i, m in enumerate(results['top_motifs'], 1):
    pos   = m['position']
    kmer  = m['kmer']
    score = m['score']
    hits  = m.get('cosmic_hits', [])

    cosmic_str = ', '.join(
        f"{h['signature']} ({h['description']})" for h in hits
    ) if hits else 'No match'

    print(f'\n  #{i}  Position {pos}-{pos+K-1}  |  k-mer: {kmer}  |  score: {score:.4f}')
    print(f'       COSMIC: {cosmic_str}')

print('\n' + '='*65)
print('Note: Matches do NOT confirm a clinical mutation.')
print('They indicate that these sequence contexts are important')
print('for the model\'s cancer prediction, and overlap with known')
print('mutational signature trinucleotide contexts.')
print('='*65)

## Step 5 — Display Saved Heatmap

In [ ]:
from IPython.display import Image, display as ipy_display
ipy_display(Image(filename=f'{SAVE_DIR}/occlusion_heatmap.png'))